# Notebook 2: Preprocessing & Manifest Building
**Autoencoders for Open-Set Presentation Attack Detection**

This notebook:
- Builds a manifest CSV of all dataset images
- Applies user-disjoint train/val/test splitting
- Samples every 5th frame to reduce dataset size
- Verifies the data pipeline with a test batch

## 1. Setup

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from src import config as cfg
from src.dataset import build_manifest, load_manifest, FacePADDataset, create_dataloaders
from src.dwt import dwt2d_numpy, visualize_dwt_bands
from src.utils import set_seed

set_seed()
cfg.print_config()

  Random seed set to 42
CONFIGURATION
  Device:            cpu
  Image Size:        128x128
  DWT Sub-band Size: 64x64
  Latent Dim:        512
  Batch Size:        16
  Epochs:            60
  Learning Rate:     0.0003
  Mixed Precision:   True
  Loss Weights:      α=0.5, β=2.0, γ=0.0
  Fusion Weight:     lambda=0.5
  Anomaly Weights:   w_iso=0.4, w_lmse=0.05, w_spec=0.4, w_maha=0.15
  IF Contamination:  0.05 (paper value)
  Train Users:       30 (IDs 1-30)
  Val Users:         7 (IDs 31-37)
  Test Users:        8 (IDs 38-45)
  Known Attacks:     [1]
  Unknown Attacks:   [2, 3, 4]


## 2. Build Manifest
Scans the entire dataset and creates a CSV with metadata for each frame.

In [2]:
manifest = build_manifest(
    dataset_dir=cfg.DATASET_DIR,
    output_path=cfg.MANIFEST_PATH,
    frame_sample_rate=cfg.FRAME_SAMPLE_RATE,
)

print(f"\nManifest saved to: {cfg.MANIFEST_PATH}")
print(f"Total entries: {len(manifest)}")

[Manifest] Total frames: 52650
  train: 35100
  val: 8190
  test: 9360
  Label 0 (Bona Fide): 5850
  Label 1 (Print (Indoor)): 11700
  Label 2 (Print (Outdoor)): 11700
  Label 3 (Screen (CCE TV)): 11700
  Label 4 (Screen (HP Monitor)): 11700
[Manifest] Saved to c:\BS_Shivang\manifest.csv

Manifest saved to: c:\BS_Shivang\manifest.csv
Total entries: 52650


## 3. Analyze Manifest

In [3]:
# Load as DataFrame for analysis
df = pd.DataFrame(manifest)
print("\n--- Split Distribution ---")
print(df.groupby(['split', 'label']).size().unstack(fill_value=0))

print("\n--- Device Distribution ---")
print(df.groupby(['device', 'label']).size().unstack(fill_value=0))

print("\n--- Category × Split ---")
print(df.groupby(['category', 'split']).size().unstack(fill_value=0))


--- Split Distribution ---
label     0     1     2     3     4
split                              
test   1040  2080  2080  2080  2080
train  3900  7800  7800  7800  7800
val     910  1820  1820  1820  1820

--- Device Distribution ---
label      0     1     2     3     4
device                              
motog5  2925  5850  5850  5850  5850
xt1572  2925  5850  5850  5850  5850

--- Category × Split ---
split          test  train   val
category                        
attack_cce     2080   7800  1820
attack_hp      2080   7800  1820
attack_print1  2080   7800  1820
attack_print2  2080   7800  1820
real           1040   3900   910


In [4]:
# Verify open-set protocol
print("\n--- Open-Set Protocol Verification ---")
train_labels = df[df['split'] == 'train']['label'].unique()
val_labels = df[df['split'] == 'val']['label'].unique()
test_labels = df[df['split'] == 'test']['label'].unique()

print(f"  Train labels: {sorted(train_labels)}")
print(f"  Val labels:   {sorted(val_labels)}")
print(f"  Test labels:  {sorted(test_labels)}")

train_users = sorted(df[df['split'] == 'train']['user_id'].unique())
val_users = sorted(df[df['split'] == 'val']['user_id'].unique())
test_users = sorted(df[df['split'] == 'test']['user_id'].unique())

print(f"\n  Train users: {train_users[0]}-{train_users[-1]} ({len(train_users)} users)")
print(f"  Val users:   {val_users[0]}-{val_users[-1]} ({len(val_users)} users)")
print(f"  Test users:  {test_users[0]}-{test_users[-1]} ({len(test_users)} users)")

# Verify no user overlap
assert len(set(train_users) & set(val_users)) == 0, "User overlap: train/val!"
assert len(set(train_users) & set(test_users)) == 0, "User overlap: train/test!"
assert len(set(val_users) & set(test_users)) == 0, "User overlap: val/test!"
print("\n  ✓ No user overlap between splits (user-disjoint)")


--- Open-Set Protocol Verification ---
  Train labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
  Val labels:   [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
  Test labels:  [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

  Train users: 1-30 (30 users)
  Val users:   31-37 (7 users)
  Test users:  38-45 (8 users)

  ✓ No user overlap between splits (user-disjoint)


## 4. Test Data Pipeline

In [5]:
# Create a small test dataset to verify the pipeline
print("\nTesting data pipeline...")
train_loader, val_loader, test_loader = create_dataloaders(manifest)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")


Testing data pipeline...
[Dataset] Split='train', bona_fide_only=True, samples=3900
[Dataset] Split='val', bona_fide_only=False, samples=8190
[Dataset] Split='test', bona_fide_only=False, samples=9360

Train batches: 243
Val batches: 512
Test batches: 585


In [6]:
# Grab one batch and inspect
batch = next(iter(train_loader))
dwt_concat, hh_band, labels, metadata = batch

print(f"\nBatch shapes:")
print(f"  DWT concat: {dwt_concat.shape}")   # Expected: (16, 12, 64, 64)
print(f"  HH band:    {hh_band.shape}")       # Expected: (16, 3, 64, 64)
print(f"  Labels:     {labels.shape}")         # Expected: (16,)
print(f"  Labels (values): {labels.tolist()}")

print(f"\nValue ranges:")
print(f"  DWT concat: [{dwt_concat.min():.4f}, {dwt_concat.max():.4f}]")
print(f"  HH band:    [{hh_band.min():.4f}, {hh_band.max():.4f}]")

c:\BS_Shivang\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Batch shapes:
  DWT concat: torch.Size([16, 12, 64, 64])
  HH band:    torch.Size([16, 3, 64, 64])
  Labels:     torch.Size([16])
  Labels (values): [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Value ranges:
  DWT concat: [-0.9392, 2.0000]
  HH band:    [-0.3294, 0.3725]


In [7]:
# Visualize a batch sample
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(min(4, dwt_concat.size(0))):
    # Show LL band (first 3 channels of dwt_concat)
    ll = dwt_concat[i, :3].permute(1, 2, 0).numpy()
    ll = (ll - ll.min()) / (ll.max() - ll.min() + 1e-8)
    axes[0, i].imshow(ll)
    axes[0, i].set_title(f'LL Band (label={labels[i].item()})')
    axes[0, i].axis('off')
    
    # Show HH band
    hh = hh_band[i].permute(1, 2, 0).numpy()
    hh = (hh - hh.min()) / (hh.max() - hh.min() + 1e-8)
    axes[1, i].imshow(hh)
    axes[1, i].set_title(f'HH Band')
    axes[1, i].axis('off')

plt.suptitle('Batch Sample: LL vs HH Bands', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(cfg.RESULTS_DIR, 'batch_sample.png'), dpi=150)
plt.show()

print("\n✓ Preprocessing pipeline verified. Proceed to Notebook 03 for training.")


✓ Preprocessing pipeline verified. Proceed to Notebook 03 for training.


C:\Users\Shivang\AppData\Local\Temp\ipykernel_26828\1359832395.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
